In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
    output_df = input_df.withColumn("ingestion_date", current_timestamp())
    return output_df

In [0]:
#def rearrange_partition_column(input_df, partition_column):
#    cols = [c for c in input_df.columns if c != partition_column]
#    cols.append(partition_column)
#    return input_df.select(*cols) 

In [0]:
import uuid

def merge_into_table(input_df, db_name, table_name, merge_condition):
    full_table_name = f"{db_name}.{table_name}"

    # Ensure target table exists
    if not spark.catalog.tableExists(full_table_name):
        raise Exception(f"Target table {full_table_name} does not exist")

    temp_view = f"source_{uuid.uuid4().hex}"
    input_df.createOrReplaceTempView(temp_view)

    spark.sql(f"""
        MERGE INTO {full_table_name} AS target
        USING {temp_view} AS source
        ON {merge_condition}
        WHEN NOT MATCHED THEN
          INSERT *
    """)

    spark.catalog.dropTempView(temp_view)